# **Babylon:  analyse de Réputation**

#Import des librairies

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

# URL de la page de critiques IMDb

In [ ]:
url = 'https://www.imdb.com/title/tt10640346/reviews/?ref_=tt_ov_rt'

# Chemin vers WebDriver Chrome

In [ ]:
driver_path = '/Users/monkeydziyech/Downloads/chromedriver_mac64/chromedriver'

# Initialisez les options du pilote Chrome

In [ ]:
chrome_options = Options()
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.binary_location = '/Applications/Google Chrome.app/Contents/MacOS/Google Chrome'  # Ajoutez le chemin vers Chrome si nécessaire


# Initialisez le WebDriver avec les options

In [ ]:
driver = webdriver.Chrome(options=chrome_options)

# Ouvrez l'URL

In [ ]:
driver.get(url)

# Attendez que la page soit initialement chargée

In [ ]:
time.sleep(2)

# Boucle pour cliquer sur le bouton "load more" jusqu'à ce qu'il n'existe plus

In [ ]:
while True:
    try:
        # Attendez que le bouton "load more" soit visible
        load_more_button = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, '.ipl-load-more__button')))
        # Exécutez un script JavaScript pour faire défiler la page jusqu'au bouton
        driver.execute_script("arguments[0].scrollIntoView();", load_more_button)
        # Attendez que le bouton "load more" soit cliquable, puis cliquez dessus
        load_more_button.click()
        # Attendez le chargement du contenu
        time.sleep(2)
    except Exception as e:
        print(f"An error occurred: {e}")
        break

# Maintenant que toutes les critiques sont chargées, analysez la page avec BeautifulSoup

In [ ]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

# Trouvez les éléments contenant les titres des critiques


In [ ]:
review_titles = soup.find_all('a', class_='title')
reviews = []

# Parcourez les éléments et imprimez le contenu textuel de chaque titre


In [ ]:
for title in review_titles:
    reviews.append(title.get_text().strip())

In [ ]:
# Fermez le navigateur
driver.quit()

In [ ]:
import pandas as pd

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from scipy.special import softmax

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
def get_labels(scores):
    class_names = ["Negative", "Neutral", "Positive"]
    return class_names[scores.tolist().index(max(scores))]

In [ ]:
import json

reviews = []

json_file_path = "avis_imdb.json"

with open(json_file_path, 'r') as json_file:
    data = json.load(json_file)
    reviews = [item["Avis"] for item in data]



In [ ]:
labels = []

for review in reviews:


  encoded_input = tokenizer(review, return_tensors='pt')
  output = model(**encoded_input)
  scores = softmax(output[0][0].detach().numpy())
  labels.append(get_labels(scores))

In [ ]:
if len(reviews) != len(labels):
    print("Error: Reviews and labels have different sizes.")
else:
    data = {'review': reviews, 'label': labels}
    df = pd.DataFrame(data)

In [ ]:
# Specify the path where you want to save the CSV file
csv_file_path = "reviews_labels.csv"  # Replace with your desired file path

# Export the DataFrame to a CSV file
df.to_csv(csv_file_path, index=False)